In [158]:
pip install langchain langchain_text_splitters langchain_mongodb pymongo unstructured[md]


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## convertir el pdf en md lo tengo en el collab que requiere bastante maquina para una birrria de pdf de 4Mb en fin.

Variables de busqueda para chunking de datos

In [159]:
headers_to_split = [("##", "H2")]
wizard_section = "conjuros de hechicero"
priest_section = "conjuros de sacerdote"
level_section = "Conjuros de nivel"
player_class = "Wizard"
level = 1

In [160]:
import unicodedata


def normalizar_texto(texto):
    # Transforma espacios ocultos (\xa0) en espacios normales y remueve rarezas
    texto_limpio = unicodedata.normalize("NFKC", texto)
    return texto_limpio.lower().strip()

## Funciones para analizar el texto y extraer los datos relevantes de cada sección de conjuros, como alcance, componentes, duración, etc. Además, se incluye una función para crear embeddings utilizando el modelo Ollama.

In [161]:
# import re
# from langchain_ollama import OllamaEmbeddings
#
# def analyze_family(page_content : str) -> str:
#     pattern = r"\(([^)]+)\)"
#     text = page_content.split("\n")[1].strip()
#     match = re.search(pattern, text)
#     if match:
#         return match.group(0)
#     return ""
#
# def result_family(page_content : str):
#     pattern = r"\(([^)]+)\)"
#     text = page_content.split("\n")[1].strip()
#     match = re.search(pattern, text)
#     return match and match.groups()
#
# def analyze_range(page_content) -> str:
#     pattern = r"Alcance\W\s*([^\n]+)"
#     text = page_content.split("\n")[2].strip()
#     match = re.search(pattern, text)
#     if match:
#         return match.group(0)
#     return ""
#
# def analyze_components(page_content : str) -> list[str]:
#     pattern = r"Componentes\W\s*([^\n]+)"
#     text = page_content.split("\n")[3].strip()
#     match = re.search(pattern, text)
#     if match:
#         return match.group(0).split(",")
#     return []
#
#
# def analyze_duration(page_content) -> str:
#     pattern = r"Duración\W\s*([^\n]+)"
#     text = page_content.split("\n")[4].strip()
#     match = re.search(pattern, text)
#     if match:
#         return match.group(0)
#     return ""
#
#
# def analyze_time_to_cast(page_content) -> str:
#     pattern = r"Tiempo de lanzamiento\W\s*([^\n]+)"
#     text = page_content.split("\n")[5].strip()
#     match = re.search(pattern, text)
#     if match:
#         return match.group(0)
#     return ""
#
#
# def analyze_area(page_content) -> str:
#     pattern = r"Área de efecto\W\s*([^\n]+)"
#     text = page_content.split("\n")[6].strip()
#     match = re.search(pattern, text)
#     if match:
#         return match.group(0)
#     return ""
#
# def analyze_salvation_throw(page_content) -> str:
#     pattern = r"Tirada de salvación\W\s*([^\n]+)"
#     text = page_content.split("\n")[7].strip()
#     match = re.search(pattern, text)
#     if match:
#         return match.group(0)
#     return ""
#
#
# def analyze_description(page_content) -> str:
#     return page_content.split("\n")[8].strip()
#
#
# def create_embedding(param) -> list[float]:
#     embeddings_model = OllamaEmbeddings(
#     model="bge-m3",
#     temperature=0,
#     top_p=0.7
#     )
#     resultado = embeddings_model.embed_documents(param)
#     return resultado[0]
#
#
#
#


## itero sobre el documento y me quedo con la primera seccion de conjuros de hechizero y conjuros de sacerdote


In [162]:
from langchain_community.document_loaders import UnstructuredMarkdownLoader, TextLoader
from langchain_text_splitters import MarkdownHeaderTextSplitter

loader = TextLoader('./hechizos.md')
docs = loader.load()
markdown_text = docs[0].page_content

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split, strip_headers=False)
sections = markdown_splitter.split_text(markdown_text)
spell_processed = []
spell_name: str | None = None

def analyze_text(param : str | None, page_content: str, player_class : str , level: int, jump: int):
        return {"name": param,
               "player_class": player_class,
               "level": level,
               "spell_family": analyze_family(page_content, jump),
               "range" : analyze_range(page_content, jump),
               "components" : analyze_components(page_content, jump),
               "duration" : analyze_duration(page_content, jump),
               "time_to_cast" : analyze_time_to_cast(page_content, jump),
               "area" : analyze_area(page_content, jump),
               "salvation_throw" : analyze_salvation_throw(page_content, jump),
               "description" : analyze_description(page_content, jump),
               "description_embedding" : create_embedding(param + " " + analyze_description(page_content, jump))
    }
import re
from langchain_ollama import OllamaEmbeddings

def analyze_family(page_content : str, jump: int) -> str:
    pattern = r"\(([^)]+)\)"
    text = page_content.split("\n")[1+jump].strip()
    match = re.search(pattern, text)
    if match:
        return match.group(0)
    return ""

def result_family(page_content : str):
    pattern = r"\(([^)]+)\)"
    text = page_content.split("\n")[1].strip()
    match = re.search(pattern, text)
    return match and match.groups()

def analyze_range(page_content, jump: int) -> str:
    pattern = r"Alcance\W\s*([^\n]+)"
    text = page_content.split("\n")[2+jump].strip()
    match = re.search(pattern, text)
    if match:
        return match.group(0)
    return ""

def analyze_components(page_content : str, jump: int) -> list[str]:
    pattern = r"Componentes\W\s*([^\n]+)"
    text = page_content.split("\n")[3+jump].strip()
    match = re.search(pattern, text)
    if match:
        return match.group(0).split(",")
    return []


def analyze_duration(page_content: str, jump: int) -> str:
    pattern = r"Duración\W\s*([^\n]+)"
    text = page_content.split("\n")[4+jump].strip()
    match = re.search(pattern, text)
    if match:
        return match.group(0)
    return ""


def analyze_time_to_cast(page_content: str, jump: int) -> str:
    pattern = r"Tiempo de lanzamiento\W\s*([^\n]+)"
    text = page_content.split("\n")[5+jump].strip()
    match = re.search(pattern, text)
    if match:
        return match.group(0)
    return ""


def analyze_area(page_content: str, jump: int) -> str:
    pattern = r"Área de efecto\W\s*([^\n]+)"
    text = page_content.split("\n")[6+jump].strip()
    match = re.search(pattern, text)
    if match:
        return match.group(0)
    return ""

def analyze_salvation_throw(page_content:str, jump:int) -> str:
    pattern = r"Tirada de salvación\W\s*([^\n]+)"
    text = page_content.split("\n")[7+jump].strip()
    match = re.search(pattern, text)
    if match:
        return match.group(0)
    return ""


def analyze_description(page_content: str, jump: int) -> str:
    return page_content.split("\n")[8+jump].strip()


def create_embedding(param) -> list[float]:
    embeddings_model = OllamaEmbeddings(
    model="bge-m3",
    temperature=0,
    top_p=0.7
    )
    resultado = embeddings_model.embed_documents(param)
    return resultado[0]


def page_content_starts_with(page_content: str) -> bool:
    return page_content.startswith("##")

for section in sections:
    if wizard_section.lower() in normalizar_texto(section.metadata.get("H2", "").lower()):
        continue
    elif priest_section.lower() in section.metadata.get("H2","").lower():
        player_class = "Priest"
        continue
    if level_section.lower() in section.metadata.get("H2","").lower():
        splited = section.page_content.split(" ")
        level = int(splited[len(splited)-1])
        continue
    if section.metadata.get("H2","").lower() not in (level_section.lower(), wizard_section.lower(), priest_section.lower()):
        if page_content_starts_with(section.page_content) and section.metadata.get("H2", "").strip() == section.page_content.split("##")[1].strip():
            spell_name = section.metadata.get("H2", "")
        else:
            if spell_name is None:
                print(section.metadata.get("H2", ""))
                spell_processed.append(analyze_text(section.metadata.get("H2", ""), section.page_content, player_class, level, 0))
            else:
                print(spell_name)
                spell_processed.append(analyze_text(spell_name, section.page_content, player_class, level, -1))
                spell_name = None



Afectar fuegos normales
Agrandar
Alarma
Amigos
Aparición
Armadura
Aura mágica de Nystul
Borrar
Caída de pluma
Cambiar el yo
Cantrip
Comprender lenguajes
Detectar magia
Detectar muertos vivientes
Disco flotante de Tenser
Dormir
Encontrar familiar
Escalada de araña
Escudo
Fuerza fantasmal
Grasa
Hechizar persona
Hipnotismo
Identificar
Leer magia
Luces danzantes
Luz
Manos ardientes
Marca de hechicero
Mensaje
Montura
Presa sacudidora
Protección contra el mal
Provocar
Proyectil mágico
Reflejo de la mirada
Reparar
Retener portal
Rociada de color
Salto
Sirviente invisible
Sonido audible
Toque helado
Ventriloquía
Alterar el yo
Asustar
Atar
Boca mágica
Bolsillos profundos
Ceguera ( Ilusión/Fantasma )
Conocer alineamiento
Detectar el mal
Detectar invisibilidad
Esfera llameante
Esquema hipnótico
Flecha ácida de Melf
Fuerza
Fuerza fantasmal mejorada
Hacer añicos
Imagen en un espejo
Incontrolable risa horrible de Tasha
Invisibilidad
Irritación
Levitar
Localizar objeto
Luz continua
Llamada a la puert

IndexError: list index out of range